In [0]:
# CELL: Install Python-Dotenv
#-----------------------------------------------------------

# 1. Install Python-Dotenv
%pip install python-dotenv

# 2. Verify Installation
try:
    from dotenv import load_dotenv
    print("Dotenv is installed and working correctly.")
except ImportError:
    raise ImportError(
        "Dotenv failed to install.  ", 
        "Please rerun this cell",
    )

In [0]:
# CELL: Setup & Load
#-----------------------------------------------------------

from dotenv import load_dotenv
import os
from pyspark.sql.functions import col, to_date, when, lit

In [0]:
# CELL: Load Environment Variables
#-----------------------------------------------------------

# 1. Load environment variables from file
load_dotenv()

# 2. Assign storage acount name and key vars from env
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
storage_account_key = os.getenv("STORAGE_ACCOUNT_KEY")

# 3. Verify env read successfully
print(f"storage_account_name = {storage_account_name}")

In [0]:
# CELL: Configure spark settings
#-----------------------------------------------------------

# 1. AUTO-CREATE-FIX: Configuration auto-creates the 'bronze' container if it is missing
spark.conf.set(
    f"fs.azure.createRemoteFileSystemDuringInitialization", "true"
)

# 2. Connect to storage account
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key,
)

print("Configuration loaded & Auto-create enabled")

In [0]:
# CELL: Assign constants
#-----------------------------------------------------------

# 1. Assign zones 
ZN_TRANSACTIONS = "transactions"
ZN_UNITS = "units"
ZN_PROJECTS = "projects"

# 1. Assign Bronze & Silver
BRONZE = "bronze"
SILVER = "silver"

# 2. Assign medallion paths
MEDALLION_PATHS = {medallion: f"abfss://{medallion}@{storage_account_name}.dfs.core.windows.net/" for medallion in [BRONZE, SILVER]}

BRONZE_PATH = MEDALLION_PATHS[BRONZE]
SILVER_PATH = MEDALLION_PATHS[SILVER]

DEST_LANDING = "landing/"
CURRENT_CSV = "current.csv"

CURRENT_CSV_PATHS_MAPPINGS = {zone: f"{BRONZE_PATH}{zone}/{DEST_LANDING}{CURRENT_CSV}" for zone in [ZN_UNITS, ZN_PROJECTS, ZN_TRANSACTIONS]}

In [0]:
# CELL: Create Bronze Input Dataframes
#-----------------------------------------------------------

# 1. Assign Bronze Dataframes mapped to appropriate zone
BRONZE_DF_MAPPINGS = {
    zone: spark.read.option("header", "true").csv(csv_path) 
    for zone, csv_path 
    in CURRENT_CSV_PATHS_MAPPINGS.items()
}


In [0]:
# CELL: Announce Configurations Setup and Data Loaded
#-----------------------------------------------------------

print(f"{BRONZE} Data Loaded and {SILVER} Container Configured")

In [0]:
# CELL: Assign as individual DataFrames
#-----------------------------------------------------------

df_bronze_txns = BRONZE_DF_MAPPINGS[ZN_TRANSACTIONS]
df_bronze_proj = BRONZE_DF_MAPPINGS[ZN_PROJECTS]
df_bronze_unit = BRONZE_DF_MAPPINGS[ZN_UNITS]


In [0]:
# CELL: Clean Projects Dimensions (Status & Dates Only)
#-----------------------------------------------------------

df_projects_clean = df_bronze_proj.select(
    col("project_number").cast("int"),
    col("project_status"),
    col("percent_completed").cast("double"),
    to_date(col("project_start_date"), "dd-MM-yyyy").alias("start_date"),
    to_date(col("completion_date"), "dd-MM-yyyy").alias("completion_date"),
    col("master_project_en").alias("master_community")).dropDuplicates(["project_number"])


print("Projects Cleaned")


In [0]:
# CELL: Display DF_Projects_Clean
#-----------------------------------------------------------

display(df_projects_clean.limit(5))

In [0]:
# CELL: Clean Transactions Fact
#-----------------------------------------------------------
df_txns_clean = df_bronze_txns.select(
    col("transaction_id"),
    to_date(col("instance_date"), "dd-MM-yyyy").alias("transaction_date"),
    col("procedure_name_en").alias("transaction_type"),
    col("trans_group_en").alias("transaction_group"),
    # Location and Project Info
    col("area_name_en").alias("area_name"),
    col("building_name_en").alias("building_name"),
    col("project_number").cast("int"),

    # Get English Names
    col("project_name_en").alias("project_name"),
    col("nearest_landmark_en").alias("landmark"),

    # Property Details
    col("property_type_en").alias("property_type"),
    col("property_sub_type_en").alias("property_subtype"),
    col("rooms_en").alias("room_type"),

    # Metrics
    col("procedure_area").cast("double").alias("area_sqft"),
    col("actual_worth").cast("double").alias("price"),
    col("meter_sale_price").cast("double")
)

# filter garbage data
df_txns_clean = df_txns_clean.filter(col("price") > 0)

print("Transactions Cleaned")

In [0]:
# CELL: Display DF_Transactions_Clean
#-----------------------------------------------------------

display(df_txns_clean.limit(5))

In [0]:
# CELL: Join & Enrich (Silver Master Table)
#-----------------------------------------------------------

# 1. Join Transactions with Projects
df_enriched = df_txns_clean.join(
    df_projects_clean,
    on="project_number",
    how="left",
)

# 2. Enrich: Calculate Price/Sq Ft
df_enriched = df_enriched.withColumn(
    "price_per_sqft",
    when(
        col("area_sqft") > 0, 
        col("price") / col("area_sqft")
    ).otherwise(0),
)

# 3. Enrich: Is Off-Plan?
df_enriched = df_enriched.withColumn(
    "is_offplan_flag",
    when(
        (col("transaction_date") < col("completion_date")) | (col("percent_completed") < 100),
        lit(True)
    ).otherwise(lit(False))
)

print("Data Enriched & Joined")


In [0]:
# CELL: Display DF_Enriched
#-----------------------------------------------------------

display(df_enriched.select(
    "transaction_date",
    "project_name",
    "room_type",
    "price",
    "is_offplan_flag",
).limit(10))

In [0]:
# CELL: Write to Silver Layer (Physical Files)
#-----------------------------------------------------------

# 1. Write the Master Transactions Table
print("Writing Transactions to Silver...")
df_enriched.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}transactions")

# 2. Write the Clean Units Table
df_units_clean = df_bronze_unit.select(
    col("property_id"),
    col("project_id").cast("int").alias("project_number"),
    col("unit_number"),

    # Metrics
    col("rooms").alias("bedrooms"),
    col("rooms_en").alias("room_type"),

    col("actual_area").cast("double").alias("unit_area"),
).dropDuplicates(["property_id"])

df_units_clean.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}units")

print("SUCCESS: Silver Files Written to Azure")

In [0]:
# CELL: Verify Written Data Exists
#-----------------------------------------------------------

print("Verifying Silver Layer...")

try: 
# 1. Load the new Delta Tables directly from the file path
    df_silver_txns = spark.read.format("delta").load(f"{SILVER_PATH}transactions")
    df_silver_unit = spark.read.format("delta").load(f"{SILVER_PATH}units")

# 2. Print Counts
    count_txns = df_silver_txns.count()
    count_unit = df_silver_unit.count()

    print(f"VERIFIED: Transactions Table contains {count_txns} rows.")
    print(f"VERIFIED: Units Table contains {count_unit} rows.")

    if count_txns > 0:
        print("Success! Here is a preview of you clean Silver data:")
        display(df_silver_txns.limit(5))
    else:
        print("WARNING: Tables exist but are empty")
except Exception as e:
    print(f"ERROR: Could not read Silver tables. Reason: {e}")